In [53]:
from _setup import setup_project_root
PROJECT_ROOT = setup_project_root()
PROJECT_ROOT


WindowsPath('C:/Users/huyy/AirPollutionPrediction-CNN-BiLSTM')

In [68]:
import importlib
import joblib
from datetime import timedelta
import models.cnn_bilstm as cnn_bilstm

importlib.reload(cnn_bilstm)

<module 'models.cnn_bilstm' from 'C:\\Users\\huyy\\AirPollutionPrediction-CNN-BiLSTM\\models\\cnn_bilstm.py'>

In [55]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent   

In [56]:
import importlib
import evaluation.metrics as metrics

importlib.reload(metrics)

<module 'evaluation.metrics' from 'C:\\Users\\huyy\\AirPollutionPrediction-CNN-BiLSTM\\evaluation\\metrics.py'>

In [80]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from evaluation.metrics import compute_metrics, compute_metrics_real_scale
from models.cnn_bilstm import (
    train_cnn_bilstm_multi,
    predict_cnn_bilstm_multi,
    build_sequences_multi,
)

In [5]:
train_path = PROJECT_ROOT / "data" / "global_split" / "train.csv"
val_path   = PROJECT_ROOT / "data" / "global_split" / "val.csv"
test_path  = PROJECT_ROOT / "data" / "global_split" / "test.csv"

train_df = pd.read_csv(train_path)
val_df   = pd.read_csv(val_path)
test_df  = pd.read_csv(test_path)

In [6]:
target_col = "PM2.5"
id_cols = ["Station_No", "date"]

In [ ]:
pm25_scaler = joblib.load(PROJECT_ROOT / "artifacts" / "pm25_scaler.pkl")

In [7]:
required_cols = set(id_cols + [target_col])
missing_required = required_cols - set(train_df.columns)
if missing_required:
    raise ValueError(f"Missing required columns in train_df: {missing_required}")

In [8]:
feature_cols = [c for c in train_df.columns if c not in (id_cols + [target_col])]

print("Target:", target_col)
print("Num features:", len(feature_cols))
print("First 10 features:", feature_cols[:10])

Target: PM2.5
Num features: 32
First 10 features: ['TSP', 'O3', 'CO', 'NO2', 'SO2', 'Temperature', 'Humidity', 'hour', 'day_of_week', 'hour_sin']


In [11]:
X_train = train_df[feature_cols].values
y_train = train_df[target_col].values

X_val = val_df[feature_cols].values
y_val = val_df[target_col].values

X_test = test_df[feature_cols].values
y_test = test_df[target_col].values

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   "y_val:  ", y_val.shape)
print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)

X_train: (55612, 32) y_train: (55612,)
X_val:   (6951, 32) y_val:   (6951,)
X_test:  (6952, 32) y_test:  (6952,)


In [59]:
df_all = pd.concat([train_df, val_df, test_df], axis=0, ignore_index=True)
df_all["date"] = pd.to_datetime(df_all["date"])
df_all = df_all.sort_values("date").reset_index(drop=True)

In [60]:
target_col = "PM2.5"
id_cols = ["Station_No", "date"]  

feature_cols = [c for c in df_all.columns if c not in (id_cols + [target_col])]

In [61]:
print("Target:", target_col)
print("Num features:", len(feature_cols))
print("First 10 features:", feature_cols[:10])
print("Date range:", df_all["date"].min(), "->", df_all["date"].max())

Target: PM2.5
Num features: 32
First 10 features: ['TSP', 'O3', 'CO', 'NO2', 'SO2', 'Temperature', 'Humidity', 'hour', 'day_of_week', 'hour_sin']
Date range: 2021-02-23 21:00:00 -> 2022-06-21 17:00:00


In [62]:
X_all = df_all[feature_cols].values.astype(np.float32)
y_all = df_all[target_col].values.astype(np.float32)

print("X_all:", X_all.shape, "y_all:", y_all.shape)

X_all: (69515, 32) y_all: (69515,)


In [65]:
TRAIN_DAYS = 90          
STEP_DAYS = 7            
TEST_RATIO = 0.10        

dates = df_all["date"].values

start_time = df_all["date"].min()
end_time   = df_all["date"].max()

windows = []
t0 = start_time

In [66]:
while True:
    train_start = t0
    train_end   = train_start + timedelta(days=TRAIN_DAYS)

    # train mask
    train_mask = (df_all["date"] >= train_start) & (df_all["date"] < train_end)
    n_train = int(train_mask.sum())

    if n_train < 2000:  
        break

    n_test = int(np.floor(n_train * TEST_RATIO))
    test_start = train_end
    test_end = test_start + timedelta(hours=n_test)  

    test_mask = (df_all["date"] >= test_start) & (df_all["date"] < test_end)
    n_test_real = int(test_mask.sum())

    if n_test_real < max(200, int(0.8 * n_test)):
        break

    windows.append({
        "train_start": train_start,
        "train_end": train_end,
        "test_start": test_start,
        "test_end": test_end,
        "n_train": n_train,
        "n_test": n_test_real
    })

    t0 = t0 + timedelta(days=STEP_DAYS)


In [67]:
print("Num windows:", len(windows))
print("First window:", windows[0] if windows else None)
print("Last  window:", windows[-1] if windows else None)

Num windows: 56
First window: {'train_start': Timestamp('2021-02-23 21:00:00'), 'train_end': Timestamp('2021-05-24 21:00:00'), 'test_start': Timestamp('2021-05-24 21:00:00'), 'test_end': Timestamp('2021-07-17 19:00:00'), 'n_train': 12941, 'n_test': 7764}
Last  window: {'train_start': Timestamp('2022-03-15 21:00:00'), 'train_end': Timestamp('2022-06-13 21:00:00'), 'test_start': Timestamp('2022-06-13 21:00:00'), 'test_end': Timestamp('2022-08-06 21:00:00'), 'n_train': 12960, 'n_test': 1134}


In [70]:
INPUT_LEN = 48
HORIZON = 24
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

pm25_scaler = joblib.load(ARTIFACTS_DIR / "pm25_scaler.pkl")

In [71]:
print("INPUT_LEN:", INPUT_LEN, "HORIZON:", HORIZON)
print("Loaded pm25_scaler:", ARTIFACTS_DIR / "pm25_scaler.pkl")

INPUT_LEN: 48 HORIZON: 24
Loaded pm25_scaler: c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\pm25_scaler.pkl


In [72]:
def flatten_multistep(y_pred_2d, y_true_2d):
    """
    Convert (n_samples, HORIZON) -> (n_samples*HORIZON,)
    so we can compute overall metrics exactly like paper's aggregate.
    """
    yp = np.asarray(y_pred_2d).reshape(-1)
    yt = np.asarray(y_true_2d).reshape(-1)
    return yt, yp

In [73]:
pretrain_end = windows[0]["train_end"]  # chỉ dùng dữ liệu trước mốc này

mask_pre = df_all["date"] < pretrain_end
X_pre = X_all[mask_pre.values]
y_pre = y_all[mask_pre.values]

In [74]:
print("Pretrain range:", df_all.loc[mask_pre, "date"].min(), "->", df_all.loc[mask_pre, "date"].max())
print("X_pre:", X_pre.shape, "y_pre:", y_pre.shape)

Pretrain range: 2021-02-23 21:00:00 -> 2021-05-24 20:00:00
X_pre: (12941, 32) y_pre: (12941,)


In [75]:
n_pre = len(X_pre)
n_pre_train = int(n_pre * 0.9)

X_pre_train, y_pre_train = X_pre[:n_pre_train], y_pre[:n_pre_train]
X_pre_val,   y_pre_val   = X_pre[n_pre_train:], y_pre[n_pre_train:]

print("Pretrain train:", X_pre_train.shape, "val:", X_pre_val.shape)

pre_ckpt = ARTIFACTS_DIR / "cnn_bilstm_pretrain_best.keras"
print("Pretrain checkpoint:", pre_ckpt)

Pretrain train: (11646, 32) val: (1295, 32)
Pretrain checkpoint: c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_pretrain_best.keras


In [76]:
pre_model, pre_hist = train_cnn_bilstm_multi(
    X_pre_train, y_pre_train,
    X_pre_val, y_pre_val,
    input_len=INPUT_LEN,
    horizon=HORIZON,
    epochs=60,               # pretrain đủ lâu
    batch_size=64,
    use_early_stopping=True,
    patience=10,
    learning_rate=3e-4,
    lstm_units=128,
    use_attention=True,
    checkpoint_path=str(pre_ckpt)
)

print("Pretrain done. ckpt exists?", pre_ckpt.exists())

Epoch 1/60
181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 113ms/step - loss: 0.0716
Epoch 1: val_loss improved from None to 0.01929, saving model to c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_pretrain_best.keras
181/181 ━━━━━━━━━━━━━━━━━━━━ 30s 122ms/step - loss: 0.0366 - val_loss: 0.0193 - learning_rate: 3.0000e-04
Epoch 2/60
181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 117ms/step - loss: 0.0176
Epoch 2: val_loss did not improve from 0.01929
181/181 ━━━━━━━━━━━━━━━━━━━━ 22s 122ms/step - loss: 0.0166 - val_loss: 0.0205 - learning_rate: 3.0000e-04
Epoch 3/60
181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - loss: 0.0147
Epoch 3: val_loss improved from 0.01929 to 0.01870, saving model to c:\Users\huyy\AirPollutionPrediction-CNN-BiLSTM\artifacts\cnn_bilstm_pretrain_best.keras
181/181 ━━━━━━━━━━━━━━━━━━━━ 23s 129ms/step - loss: 0.0143 - val_loss: 0.0187 - learning_rate: 3.0000e-04
Epoch 4/60
181/181 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - loss: 0.0134
Epoch 4: val_loss improved from 0.01870 to 0.018

In [78]:
pre_ckpt = ARTIFACTS_DIR / "cnn_bilstm_pretrain_best.keras"
base_model = load_model(pre_ckpt)
results = []

In [ ]:
FINE_EPOCHS = 20
FINE_LR = 5e-5

In [ ]:
for i, w in enumerate(windows, start=1):
    tr_mask = (df_all["date"] >= w["train_start"]) & (df_all["date"] < w["train_end"])
    te_mask = (df_all["date"] >= w["test_start"])  & (df_all["date"] < w["test_end"])

    Xtr, ytr = X_all[tr_mask.values], y_all[tr_mask.values]
    Xte, yte = X_all[te_mask.values], y_all[te_mask.values]

    # skip if too short for 48->24
    if len(Xtr) < (INPUT_LEN + HORIZON + 200) or len(Xte) < (INPUT_LEN + HORIZON + 50):
        continue

    # time split inside window: train/val (90/10)
    n_tr = len(Xtr)
    n_tr_train = int(n_tr * 0.9)

    Xtr_train, ytr_train = Xtr[:n_tr_train], ytr[:n_tr_train]
    Xtr_val,   ytr_val   = Xtr[n_tr_train:], ytr[n_tr_train:]

    model_w = tf.keras.models.clone_model(base_model)
    model_w.set_weights(base_model.get_weights())

    model_w.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=FINE_LR, clipnorm=1.0),
        loss=tf.keras.losses.Huber(delta=1.0)
    )
    callbacks_ft = [
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=0
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=6, restore_best_weights=True
    )
    ]


    # build sequences
    Xtr_seq, ytr_seq = build_sequences_multi(Xtr_train, ytr_train, input_len=INPUT_LEN, horizon=HORIZON)
    Xval_seq, yval_seq = build_sequences_multi(Xtr_val, ytr_val, input_len=INPUT_LEN, horizon=HORIZON)

    model_w.fit(
    Xtr_seq, ytr_seq,
    validation_data=(Xval_seq, yval_seq),
    epochs=FINE_EPOCHS,
    batch_size=64,
    verbose=0,
    shuffle=False,          
    callbacks=callbacks_ft  
    )

    y_pred_2d = predict_cnn_bilstm_multi(model_w, Xte, input_len=INPUT_LEN, horizon=HORIZON)

    _, y_true_2d = build_sequences_multi(Xte, yte, input_len=INPUT_LEN, horizon=HORIZON)

    y_true_flat, y_pred_flat = flatten_multistep(y_pred_2d, y_true_2d)

    m = compute_metrics_real_scale(
        y_true_scaled=y_true_flat,
        y_pred_scaled=y_pred_flat,
        scaler=pm25_scaler,
        eps=1.0,
        mask_nan=True
    )

    m["window_id"] = i
    m["n_test_points"] = int(y_true_flat.shape[0])
    results.append(m)

    if i % 5 == 0:
        print(f"Done window {i}/{len(windows)} | Corr={m['Correlation']:.3f} RMSE={m['RMSE']:.2f} MAE={m['MAE']:.2f} MAPE={m['MAPE']:.3f}")

results_df = pd.DataFrame(results)
print("\nWindows evaluated:", len(results_df))
results_df.head()

Done window 5/56 | Corr=0.357 RMSE=56.30 MAE=46.60 MAPE=0.513
Done window 10/56 | Corr=0.123 RMSE=58.77 MAE=42.84 MAPE=0.369
Done window 15/56 | Corr=0.309 RMSE=76.98 MAE=56.84 MAPE=0.547
Done window 20/56 | Corr=0.345 RMSE=81.90 MAE=63.51 MAPE=0.539
Done window 25/56 | Corr=0.296 RMSE=82.17 MAE=63.50 MAPE=0.412
Done window 30/56 | Corr=0.538 RMSE=75.81 MAE=58.67 MAPE=0.414
Done window 35/56 | Corr=0.612 RMSE=64.05 MAE=50.04 MAPE=0.421
Done window 40/56 | Corr=0.634 RMSE=65.32 MAE=49.32 MAPE=0.453
Done window 45/56 | Corr=0.583 RMSE=61.96 MAE=45.18 MAPE=0.451
Done window 50/56 | Corr=0.410 RMSE=64.79 MAE=47.53 MAPE=3.872
Done window 55/56 | Corr=0.491 RMSE=69.64 MAE=50.97 MAPE=1.567

Windows evaluated: 62


,Correlation,RMSE,MAPE,MAE,window_id,n_test_points
0,0.330472,57.228217,0.380684,40.776032,1,184608
1,0.411061,51.093011,0.295832,33.975383,2,184896
2,0.437883,53.172127,0.436000,41.382518,3,184896
3,0.378278,51.547591,0.398253,39.306942,4,184896
4,0.374070,48.099873,0.385307,37.119368,5,184896


In [83]:
print("len(windows):", len(windows))
print("len(results):", len(results))
print("results_df rows:", len(results_df))
print("unique window_id:", results_df["window_id"].nunique())
print("max window_id:", results_df["window_id"].max())
print("duplicated window_id rows:", results_df["window_id"].duplicated().sum())


len(windows): 56
len(results): 62
results_df rows: 62
unique window_id: 56
max window_id: 56
duplicated window_id rows: 6


In [85]:
results = []
try:
    del results_df
except NameError:
    pass

print("Reset done.")

Reset done.
